# Notebook For Analysis

# Setup

In [1]:
!echo $HOSTNAME
!python --version
!nvidia-smi

g006
Python 3.12.5
Thu Jun 25 21:00:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 575.57.08              Driver Version: 575.57.08      CUDA Version: 12.9     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-PCIE-40GB          Off |   00000000:17:00.0 Off |                    0 |
| N/A   24C    P0             35W /  250W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+----------------------------

In [2]:
# General Libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import einops
from fancy_einsum import einsum
import os
import sys
import tqdm.auto as tqdm
import random
from pathlib import Path
import plotly.express as px
import copy

from typing import List, Union, Optional
from functools import partial
import itertools
from IPython.display import HTML

# CSV Use Libraries
import pandas as pd

# Function Imports
from torch.optim.lr_scheduler import ReduceLROnPlateau

# Unused
# from torch.utils.data import DataLoader
# from transformers import AutoModelForCausalLM, AutoConfig, AutoTokenizer
# import dataclasses
# import datasets
# import ast
# from transformers import get_cosine_schedule_with_warmup #Used for lr?
# from torch.nn.utils import clip_grad_norm_

In [3]:
from config import *

# Shared helpers live in Transformer.py (the training script). Importing them does
# NOT trigger training — that is guarded by `if __name__ == "__main__"`.
from Transformer import (
    setup_device, build_model, load_and_split_data, load_checkpoint_into,
    descent_loss_fn, descent_accuracy_fn, descent_sequence_accuracy,
    create_attention_mask, register_pad_mask_hook,
)

Data Dir:  /home/linrya/CoxeterArtinProject/transformer/2026/Categorical_Transformer
Model PTH: /home/linrya/CoxeterArtinProject/transformer/2026/Categorical_Transformer/workspace/_scratch/model.pth


## Define Graphing Functs

In [4]:
import plotly.io as pio
pio.renderers.default = "notebook_connected"
print(f"Using renderer: {pio.renderers.default}")

Using renderer: notebook_connected


In [5]:
pio.templates['plotly'].layout.xaxis.title.font.size = 20
pio.templates['plotly'].layout.yaxis.title.font.size = 20
pio.templates['plotly'].layout.title.font.size = 30

In [6]:
import transformer_lens
import transformer_lens.utilities as utils
from transformer_lens import HookedTransformer
import transformer_lens.config.hooked_transformer_config as htc
sys.modules['transformer_lens.HookedTransformerConfig'] = sys.modules['transformer_lens.config.hooked_transformer_config']

In [7]:
# Unused plotting functions in favor of Neel Plotly

# def imshow(tensor, renderer=None, xaxis="", yaxis="", **kwargs):
#     px.imshow(utils.to_numpy(tensor), color_continuous_midpoint=0.0, color_continuous_scale="RdBu", labels={"x":xaxis, "y":yaxis}, **kwargs).show(renderer)

# def line(tensor, renderer=None, xaxis="", yaxis="", **kwargs):
#     px.line(utils.to_numpy(tensor), labels={"x":xaxis, "y":yaxis}, **kwargs).show(renderer)

# def scatter(x, y, xaxis="", yaxis="", caxis="", renderer=None, **kwargs):
#     x = utils.to_numpy(x)
#     y = utils.to_numpy(y)
#     px.scatter(y=y, x=x, labels={"x":xaxis, "y":yaxis, "color":caxis}, **kwargs).show(renderer)

## Set Paths and GPU Devices

In [10]:
# PTH_LOCATION is imported from config.py; create the directory if it doesn't exist yet
# DATA_PATH and PTH_LOCATION are now resolved from config.py
os.makedirs(Path(PTH_LOCATION).parent, exist_ok=True)
print(f"Data Dir:  {DATA_PATH}")
print(f"Model PTH: {PTH_LOCATION}")

Data Dir:  /home/linrya/CoxeterArtinProject/transformer/2026/Categorical_Transformer
Model PTH: /home/linrya/CoxeterArtinProject/transformer/2026/Categorical_Transformer/workspace/_scratch/model.pth


In [11]:
device, device1 = setup_device()

CUDA available: True
Device count: 1
Device 0: NVIDIA A100-PCIE-40GB
Device Name: cuda


## Define Model and Optimizer Params

In [12]:
torch.manual_seed(seed=DATA_SEED)
torch.cuda.manual_seed_all(DATA_SEED)

# Load architecture + weights from the saved checkpoint. build_model() reconstructs
# the model/optimizer/scheduler (and disables biases); load_checkpoint_into() fills in
# the trained state and returns the training-curve history.
cached_data = torch.load(PTH_LOCATION, weights_only=False)
cfg = cached_data["config"]

model, optimizer, scheduler = build_model(cfg)
_, history = load_checkpoint_into(model, optimizer, scheduler)

model_checkpoints = history["checkpoints"]
checkpoint_epochs = history["checkpoint_epochs"]
train_losses      = history["train_losses"]
test_losses       = history["test_losses"]
train_accuracies  = history["train_accuracies"]
test_accuracies   = history["test_accuracies"]

## Misc Setup
Loss/accuracy/masking functions are imported from `Transformer.py` at the top.

In [33]:
torch.cuda.empty_cache()
clsDex = 21 # index of classification token, kept for attention pattern analysis

## Initialize Datasets

In [14]:
# Load + shuffle + split + move-to-device, using the same seeded split as training.
data = load_and_split_data(device1)
train_tokens,  test_tokens  = data["train_tokens"],  data["test_tokens"]
train_targets, test_targets = data["train_targets"], data["test_targets"]
train_mask,    test_mask    = data["train_mask"],    data["test_mask"]
train_attention_mask = data["train_attention_mask"]
test_attention_mask  = data["test_attention_mask"]

Loaded dataset: tokens (10000, 22) | targets (10000, 22, 3)
Train size: 4000 | Test size: 6000 | Seq Len: 22


# Graph Results

In [15]:
from neel_plotly.plot import line_or_scatter, line as neel_line

In [ ]:
print(CHECKPOINT_STEP)
skipBy = 1


def createGraph(yTrain, yTest, yName, title):
    fig = line_or_scatter(
        [yTrain[::skipBy], yTest[::skipBy]],
        x=np.arange(0, len(yTrain), skipBy),
        xaxis="Epoch",
        yaxis=yName,
        log_y=True,
        title=title,
        line_labels=['train', 'test'],
        toggle_x=True,
        toggle_y=True,
        plot_type="line",
        return_fig=True
    )
    return fig

fig1 = createGraph(train_losses, test_losses, "Loss", "Loss Curve for Word Problem")
fig2 = createGraph(train_accuracies, test_accuracies, "Accuracy", "Accuracy Curve for Word Problem")

fig1.show()
fig2.show()

fig1.write_html("loss_curve.html")
fig2.write_html("accuracy_curve.html")

# Analysing the Model

Helpful Memory Probing Functions:

- nvidia-smi
- torch.cuda.empty_cache()
- torch.set_grad_enabled(mode=False)
- gc.collect()
- print(torch.cuda.memory_summary())
- torch profiler also (saved image on Aug 03, 2025 ~6pm)

### Helper Functions
- logit return function
- prediction function

In [18]:
import gc

def getLogits(model: HookedTransformer, tokens, getCache: bool = False):
    """
    Applies the padding mask and runs a forward pass.
    Returns detached logits (and optionally the activation cache).
    """
    with torch.inference_mode():
        model.reset_hooks()
        mask = create_attention_mask(tokens).to(device1)
        register_pad_mask_hook(model, mask)
        if getCache:
            original_logits_og, cache = model.run_with_cache(tokens)
        else:
            original_logits_og = model(tokens)
    original_logits = original_logits_og.detach().clone()
    del original_logits_og, mask
    torch.cuda.empty_cache()
    if getCache:
        return original_logits, cache
    else:
        return original_logits

def getPredictions(model: HookedTransformer, tokens, targets, mask):
    """
    Per-sequence descent accuracy: fraction of prefixes whose descent set is
    exactly correct. Shape: (batch_size,)
    """
    logits  = getLogits(model, tokens, getCache=False)
    seq_acc = descent_sequence_accuracy(logits, targets, mask)
    del logits
    return seq_acc

def imshow(tensor, renderer=None, xaxis="", yaxis="", xlabels=None, ylabels=None, aspect="auto", **kwargs):
    fig = px.imshow(
        utils.to_numpy(tensor),
        color_continuous_midpoint=0.0,
        color_continuous_scale="RdBu",
        labels={"x": xaxis, "y": yaxis},
        aspect=aspect,
        **kwargs
    )
    if xlabels is not None:
        fig.update_xaxes(tickmode='array', tickvals=list(range(len(xlabels))), ticktext=xlabels)
    if ylabels is not None:
        fig.update_yaxes(tickmode='array', tickvals=list(range(len(ylabels))), ticktext=ylabels)
    fig.update_yaxes(scaleanchor=None)
    fig.show(renderer)
    return fig

def cleanup():
    gc.collect()
    torch.cuda.empty_cache()

## Quick Analysis + Memory Cleanup

In [19]:
torch.set_grad_enabled(False)
gc.collect()
torch.cuda.empty_cache()

In [20]:
original_logits, cache = getLogits(model, train_tokens, getCache=True)

Get key weight matrices:

In [21]:
W_E = model.embed.W_E[:-1]
print("W_E", W_E.shape)
W_neur = W_E @ model.blocks[0].attn.W_V @ model.blocks[0].attn.W_O @ model.blocks[0].mlp.W_in
print("W_neur", W_neur.shape)
W_logit = model.blocks[0].mlp.W_out @ model.unembed.W_U
print("W_logit", W_logit.shape)

W_E torch.Size([3, 256])
W_neur torch.Size([4, 3, 256])
W_logit torch.Size([256, 3])


In [22]:
original_loss = descent_loss_fn(original_logits, train_targets, train_mask).item()
print("Original Loss:", original_loss)

Original Loss: 0.3704884350299835


### Looking at Activations

Get all shapes:

In [23]:
for param_name, param in cache.items():
    print(param_name, param.shape)

hook_embed torch.Size([4000, 22, 256])
hook_pos_embed torch.Size([4000, 22, 256])
blocks.0.hook_resid_pre torch.Size([4000, 22, 256])
blocks.0.attn.hook_q torch.Size([4000, 22, 4, 64])
blocks.0.attn.hook_k torch.Size([4000, 22, 4, 64])
blocks.0.attn.hook_v torch.Size([4000, 22, 4, 64])
blocks.0.attn.hook_attn_scores torch.Size([4000, 4, 22, 22])
blocks.0.attn.hook_pattern torch.Size([4000, 4, 22, 22])
blocks.0.attn.hook_z torch.Size([4000, 22, 4, 64])
blocks.0.hook_attn_out torch.Size([4000, 22, 256])
blocks.0.hook_resid_mid torch.Size([4000, 22, 256])
blocks.0.mlp.hook_pre torch.Size([4000, 22, 256])
blocks.0.mlp.hook_post torch.Size([4000, 22, 256])
blocks.0.hook_mlp_out torch.Size([4000, 22, 256])
blocks.0.hook_resid_post torch.Size([4000, 22, 256])
unembed.hook_in torch.Size([4000, 22, 256])
unembed.hook_out torch.Size([4000, 22, 3])


In [25]:
# debug: prints the first 4 input train data values
print(train_tokens[:4])
print(model.W_E.shape)

tensor([[3, 2, 2, 1, 2, 2, 3, 3, 1, 3, 2, 3, 2, 3, 1, 3, 2, 3, 2, 1, 2, 3],
        [1, 1, 1, 1, 3, 3, 3, 3, 2, 3, 1, 2, 1, 1, 1, 1, 3, 2, 1, 1, 1, 1],
        [2, 2, 3, 1, 2, 2, 1, 3, 1, 1, 2, 2, 2, 3, 3, 1, 3, 1, 2, 2, 3, 1],
        [2, 1, 3, 3, 1, 3, 3, 3, 3, 3, 3, 3, 3, 1, 1, 3, 3, 3, 1, 2, 3, 3]],
       device='cuda:0')
torch.Size([4, 256])


## Attention Heads (average)

### Average attention over all words for each head
Note: all train data input words used and averaged out for these graphs
- x: letters (keys attended to)
- y: letters (query giving attention to x axis)

In [26]:
n_heads = model.cfg.n_heads
seq_len = model.cfg.n_ctx
str_tokens = [str(i) for i in range(seq_len)]

with torch.inference_mode():
    for layer in range(model.cfg.n_layers):
        head_sums  = None   # reset per layer
        num_samples = 0

        for wordIndex in range(len(train_tokens)):
            # Grab attention pattern: [n_heads, seq_len, seq_len]
            attn_patterns = cache["pattern", layer][wordIndex]

            if head_sums is None:
                head_sums = torch.zeros_like(attn_patterns)

            head_sums   += attn_patterns
            num_samples += 1

        # Average attention
        avg_attn = head_sums / num_samples

        for head in range(n_heads):
            imshow(
                avg_attn[head].detach().cpu(),
                x=str_tokens,
                y=str_tokens,
                xaxis="Key (Attended To)",
                yaxis="Query (Paying Attention)",
                title=f"Layer {layer} Head {head} — Average Attention Across Dataset",
                aspect=None,
            )

### Average Attention Pattern over Attention Heads

In [34]:
for layer in range(model.cfg.n_layers):
    attention = cache["pattern", layer].mean(dim=0)[:, clsDex, :]
    imshow(
        attention,
        title=f"Average Attention Paid | for token {clsDex} | per head | layer {layer}",
        xaxis="Source",
        yaxis="Head",
        x=[str(i) for i in range(train_tokens.shape[1])],
        ylabels=[f"{i}" for i in range(attention.shape[0])]
    )

## Attention Heads (word: X)

In [59]:
# Look at a specific word in the dataset by its (shuffled) index
wordIndex = 100

# Evaluate the model on a single word
wordTensor  = train_tokens[wordIndex]
wordTokens  = wordTensor.unsqueeze(0)                  # add batch dimension
wordTargets = train_targets[wordIndex].unsqueeze(0)
wordMask    = train_mask[wordIndex].unsqueeze(0)
seq_acc = getPredictions(model, wordTokens, wordTargets, wordMask)

logits = getLogits(model, wordTokens, getCache=False)
probs = torch.sigmoid(logits)
preds = (probs > 0.5).int()

print(f"Word:    {wordTensor.tolist()}")

for t in range(wordTokens.shape[1]):
    true_labels = wordTargets[0, t].int().tolist()
    pred_labels = preds[0, t].tolist()

    print(f"{t:02d}  True: {true_labels}   Pred: {pred_labels}   Correct:{true_labels == pred_labels}")

print(f"Seq Acc: {seq_acc.item():.4f}  (fraction of prefixes with an exactly-correct descent set)")

Word:    [1, 1, 2, 2, 1, 1, 3, 2, 3, 3, 3, 2, 3, 1, 2, 3, 1, 1, 2, 1, 1, 2]
00  True: [1, 0, 0]   Pred: [1, 0, 0]   Correct:True
01  True: [0, 0, 0]   Pred: [0, 0, 0]   Correct:True
02  True: [0, 1, 0]   Pred: [0, 1, 0]   Correct:True
03  True: [0, 0, 0]   Pred: [0, 0, 0]   Correct:True
04  True: [1, 0, 0]   Pred: [1, 0, 0]   Correct:True
05  True: [0, 0, 0]   Pred: [0, 0, 0]   Correct:True
06  True: [0, 0, 1]   Pred: [0, 0, 1]   Correct:True
07  True: [0, 1, 0]   Pred: [0, 1, 0]   Correct:True
08  True: [0, 1, 1]   Pred: [0, 1, 1]   Correct:True
09  True: [0, 1, 0]   Pred: [0, 1, 0]   Correct:True
10  True: [0, 1, 1]   Pred: [0, 1, 1]   Correct:True
11  True: [0, 0, 1]   Pred: [0, 0, 1]   Correct:True
12  True: [0, 1, 0]   Pred: [0, 1, 1]   Correct:False
13  True: [1, 0, 0]   Pred: [1, 0, 0]   Correct:True
14  True: [1, 1, 0]   Pred: [1, 1, 0]   Correct:True
15  True: [0, 0, 1]   Pred: [0, 0, 1]   Correct:True
16  True: [1, 0, 1]   Pred: [1, 0, 0]   Correct:False
17  True: [0, 0, 1]  

### Attention patterns per each head on ONE word
- prints 4 graphs

In [ ]:
# Prints 1 graph per attention head
# x: letters (keys attended to)
# y: letters (query giving attention to x axis)


# Move to test device, because it has more memory
chosenWord = train_tokens[wordIndex].unsqueeze(0).to(device1)  # shape (1, 22)
print(chosenWord)


# Generate labeled tokens with position

# ex: word len 22: str_tokens = [0,...,21]
str_tokens = [f"{i}" for i, tok in enumerate(chosenWord[0])]

# Visualize
n_heads = model.cfg.n_heads
n_layers = model.cfg.n_layers
for layer in range(n_layers):
    for head in range(n_heads):
        # gets the attention pattern for layer 0, pick 1 word from the batch and look at 1 head for it
        # detatch tensor from computation graph (no gradients), move tensor to cpu to make imshow heatmap
        attn_single_head = cache["pattern", layer][wordIndex, head].detach().cpu()
        imshow(
            attn_single_head,
            x=str_tokens,
            y=str_tokens,
            xaxis="Key (Attended To)",
            yaxis="Query (Paying Attention)",
            title=f"Layer {layer} Head {head} Attention Pattern (Positional Tokens)",
            aspect=None
        )


### Attention Pattern for wordIndex over attention Heads

In [ ]:
# Attention pattern for a specific word (x: generators, y: attention heads)
# Assuming train_tokens is shape (batch, seq_len), and you're using batch 0?

token_strs = [str(x) for x in chosenWord.squeeze(0).tolist()]

for layer in range(model.cfg.n_layers):
    attention = cache["pattern", layer][wordIndex][:, clsDex, :]
    imshow(
        attention,
        title=f"Attention Pattern (for word {wordIndex}) over Attention Heads in layer {layer}",
        xaxis="Source",
        yaxis="Head",
        #x=[str(i) for i in range(train_tokens.shape[1])],
        xlabels=token_strs,
        ylabels=[f"{i}" for i in range(attention.shape[0])]
    )

print([str(i) for i in range(train_tokens.shape[1])])
print(token_strs)

## Misclassification Graphs
- Experimenting with looking at length of consistently incorrect words

### Train Misclassification Graphs:

In [ ]:
from collections import defaultdict
import matplotlib.pyplot as plt

words = []
seq_accuracies = []
lengths = []
dataset_indices = []

total_by_length         = defaultdict(int)
misclassified_by_length = defaultdict(int)

# Per-sequence descent accuracy for all train examples (1.0 = every prefix's descent set correct)
train_seq_acc = getPredictions(model, train_tokens, train_targets, train_mask)

for i in range(len(train_tokens)):
    input_seq = train_tokens[i]
    word_str  = ' '.join([str(int(tok)) for tok in input_seq.tolist()])
    word_len  = (input_seq != 0).sum().item()   # no special token in the new format

    words.append(word_str)
    seq_accuracies.append(train_seq_acc[i].item())
    lengths.append(word_len)
    dataset_indices.append(i)

    total_by_length[word_len] += 1
    if train_seq_acc[i].item() < 1.0:
        misclassified_by_length[word_len] += 1

In [ ]:
print(misclassified_by_length)

In [ ]:
# Collect imperfect sequences (seq accuracy < 1.0)
imperfect = [
    (words[i], lengths[i], seq_accuracies[i], dataset_indices[i])
    for i in range(len(words)) if seq_accuracies[i] < 1.0
]

print("\nRandom Sample of Imperfect Train Sequences:")
for word, length, acc, idx in random.sample(imperfect, min(20, len(imperfect))):
    print(f"Word: {word} | Length: {length} | Seq Acc: {acc:.3f} | Index: {idx}")

# Plot 1: Raw length distribution of imperfect sequences
imperfect_lengths = [length for _, length, _, _ in imperfect]
plt.figure(figsize=(10, 6))
plt.hist(imperfect_lengths, bins=range(min(imperfect_lengths), max(imperfect_lengths)+2), edgecolor='black')
plt.title("Length Distribution of Imperfect Train Sequences (Post Training)")
plt.xlabel("Word Length")
plt.ylabel("Number of Sequences")
plt.grid(True)
plt.tight_layout()
plt.show()

# Plot 2: Percentage imperfect per length
lengths_sorted        = sorted(total_by_length.keys())
percent_misclassified = [100 * misclassified_by_length[l] / total_by_length[l] for l in lengths_sorted]
misclassified_counts  = [misclassified_by_length[l] for l in lengths_sorted]

plt.figure(figsize=(10, 6))
bars = plt.bar(lengths_sorted, percent_misclassified, edgecolor='black')

for bar, count in zip(bars, misclassified_counts):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2, height + 1, f'{count}',
             ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.title("Percentage of Imperfect Sequences by Length (Train)")
plt.xlabel("Word Length")
plt.ylabel("Imperfect Rate (%)")
plt.xticks(lengths_sorted)
plt.grid(True)
plt.tight_layout()
plt.show()

### Test Misclassification Graphs

In [ ]:
words = []
seq_accuracies = []
lengths = []
dataset_indices = []

total_by_length         = defaultdict(int)
misclassified_by_length = defaultdict(int)

test_tokens   = test_tokens.to(device1)
test_seq_acc  = getPredictions(model, test_tokens, test_targets, test_mask)

for i in range(len(test_tokens)):
    input_seq = test_tokens[i]
    word_str  = ' '.join([str(int(tok)) for tok in input_seq.tolist()])
    word_len  = (input_seq != 0).sum().item()   # no special token in the new format

    words.append(word_str)
    seq_accuracies.append(test_seq_acc[i].item())
    lengths.append(word_len)
    dataset_indices.append(i)

    total_by_length[word_len] += 1
    if test_seq_acc[i].item() < 1.0:
        misclassified_by_length[word_len] += 1

imperfect = [
    (words[i], lengths[i], seq_accuracies[i], dataset_indices[i])
    for i in range(len(words)) if seq_accuracies[i] < 1.0
]

print("\nRandom Sample of Imperfect Test Sequences:")
for word, length, acc, idx in random.sample(imperfect, min(20, len(imperfect))):
    print(f"Word: {word} | Length: {length} | Seq Acc: {acc:.3f} | Index: {idx}")

# Plot 1: Raw length distribution of imperfect sequences
imperfect_lengths = [length for _, length, _, _ in imperfect]
plt.figure(figsize=(10, 6))
plt.hist(imperfect_lengths, bins=range(min(imperfect_lengths), max(imperfect_lengths)+2), edgecolor='black')
plt.title("Length Distribution of Imperfect Test Sequences (Post Training)")
plt.xlabel("Word Length")
plt.ylabel("Number of Sequences")
plt.grid(True)
plt.tight_layout()
plt.show()

# Plot 2: Percentage imperfect per length
lengths_sorted        = sorted(total_by_length.keys())
percent_misclassified = [100 * misclassified_by_length[l] / total_by_length[l] for l in lengths_sorted]
misclassified_counts  = [misclassified_by_length[l] for l in lengths_sorted]

plt.figure(figsize=(10, 6))
bars = plt.bar(lengths_sorted, percent_misclassified, edgecolor='black')

for bar, count in zip(bars, misclassified_counts):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2, height + 1, f'{count}',
             ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.title("Percentage of Imperfect Sequences by Length (Test)")
plt.xlabel("Word Length")
plt.ylabel("Imperfect Rate (%)")
plt.xticks(lengths_sorted)
plt.grid(True)
plt.tight_layout()
plt.show()

### Other Graphs
- check on misclassified
- attention heads of random selection of misclassified train inputs

In [ ]:
print(f"total word count: {sum(total_by_length.values())}")
print(f"total misclassified count: {sum(misclassified_by_length.values())}")

In [ ]:
# Prints 1 graph per attention head
# x: letters (keys attended to)
# y: letters (query giving attention to x axis)

# Make sure there are at least 4 items to sample from
sample_size = min(4, len(misclassified))

# Take random sample of misclassified items
sample = random.sample(misclassified, sample_size)

misclassified_index = [index for _, _, _, index in sample]
# Move to test device, because it has more memory
for mis_index in misclassified_index:
    chosenWord = test_tokens[mis_index].unsqueeze(0).to(device1)  # shape (1, 22)


    # Generate labeled tokens with position

    # ex: word len 22: str_tokens = [0,...,21]
    str_tokens = [f"{i}" for i, tok in enumerate(chosenWord[0])]

    # Visualize
    n_heads = model.cfg.n_heads
    n_layers = model.cfg.n_layers
    for layer in range(n_layers):
        for head in range(n_heads):
            # gets the attention pattern for layer 0, pick 1 word from the batch and look at 1 head for it
            # detatch tensor from computation graph (no gradients), move tensor to cpu to make imshow heatmap
            attn_single_head = cache["pattern", layer][mis_index, head].detach().cpu()
            
            imshow(
                attn_single_head,
                x=str_tokens,
                y=str_tokens,
                xaxis="Key (Attended To)",
                yaxis="Query (Paying Attention)",
                title=f"Layer {layer} Head {head} Attention Pattern (for word {mis_index})(Positional Tokens)"
            )